# Red Shift Smoke Test

Validate the OpenEnv API, SRE tools, reward breakdown, and curriculum reset path.

In [ ]:
import os, sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

from oncallenv import OnCallRedShiftEnv
from oncallenv.core.types import Action

In [ ]:
env = OnCallRedShiftEnv()
obs = env.reset(task_id="seed_easy_memory_leak")
print(obs.task_id, obs.alerts[0].message)
for command in ["kubectl_get_pods", "kubectl_logs payment-service", "kubectl_top payment-service", "kubectl_rollout_restart payment-service", "declare_resolved"]:
    obs = env.step(Action(command=command))
    print("\n$", command)
    print(obs.last_action_result[:600])
print("reward", obs.reward, obs.reward_breakdown)

In [ ]:
rca = {
    "root_cause_service": "payment-service",
    "root_cause_category": "oom_kill",
    "timeline": [{"timestamp": "2026-04-24T09:00:00Z", "service": "payment-service", "description": "OOMKilled observed"}],
    "five_whys": ["payment-service exhausted heap under transaction load", "unbounded cache retained transaction objects", "release lacked heap growth guardrails"],
    "action_items": ["Add heap alerts and cache limits for payment-service"],
    "evidence_citations": [{"source": "log", "ref": "kubectl_logs payment-service", "excerpt": "OOMKilled exit code 137"}],
    "blast_radius_description": "Checkout requests saw 503s until restart.",
}
obs = env.step(Action(command="submit_rca " + json.dumps(rca)))
print(obs.reward)
print(obs.reward_breakdown)

In [ ]:
env = OnCallRedShiftEnv()
obs = env.reset(task_id="curriculum")
print("curriculum task:", obs.task_id)
print(obs.alerts[0].message)